# 1. Normalization and Scaling
## Project: Global Life Expectancy Paradox

**Purpose:** Prepare numeric features so that measurements with very different units (population, GDP per capita, rates and percentages) can be compared by machine-learning algorithms.

**Concepts**
- **Standardization (z-score):** subtract the feature mean and divide by its standard deviation. Values are centered around 0 with standard deviation near 1.
- **Min-max scaling:** rescale a feature to the 0–1 range.
- Scaling does not change the meaning of the original data when originals are retained; it changes the numerical representation.

**Input:** the already-cleaned country-level CSV. Run the upload cell and select `CLEANED_DATASET_FINAL(1).csv`.

In [ ]:
# In Google Colab, upload CLEANED_DATASET_FINAL(1).csv when prompted.
from google.colab import files
uploaded = files.upload()
import io, pandas as pd, numpy as np
filename = next(iter(uploaded))
df = pd.read_csv(io.BytesIO(uploaded[filename]))
print("Dataset shape:", df.shape)
display(df.head())

## Step 1 — Select numeric features
Identifiers are not treated as measurements. This notebook excludes ISO codes and country names from scaling and only scales numeric columns.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler
id_cols = [c for c in ['name', 'iso2', 'region'] if c in df.columns]
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
print(f"Rows: {len(df)} | numeric features: {len(numeric_cols)}")
print("Numeric features:", numeric_cols)

## Step 2 — Standardize and normalize
The notebook creates separate columns with suffixes `_zscore` and `_minmax`. Original columns are preserved so the transformations can be checked and interpreted.

In [ ]:
z = StandardScaler()
mm = MinMaxScaler()
z_values = z.fit_transform(df[numeric_cols])
mm_values = mm.fit_transform(df[numeric_cols])
z_df = pd.DataFrame(z_values, columns=[c + '_zscore' for c in numeric_cols], index=df.index)
mm_df = pd.DataFrame(mm_values, columns=[c + '_minmax' for c in numeric_cols], index=df.index)
scaled_df = pd.concat([df.copy(), z_df, mm_df], axis=1)
display(scaled_df[[numeric_cols[0], numeric_cols[0]+'_zscore', numeric_cols[0]+'_minmax']].head())
print("Min-max range check:", float(mm_values.min()), float(mm_values.max()))

## Step 3 — Export
**Interpretation:** use z-scores for algorithms sensitive to scale; use min-max columns when a bounded 0–1 range is convenient. For predictive evaluation, fit scalers only on the training split and transform the test split to avoid data leakage.

In [ ]:
output_name = 'life_expectancy_scaled_features.csv'
scaled_df.to_csv(output_name, index=False)
from google.colab import files
files.download(output_name)